#  UK Electronics Scraper — Selenium Edition


---

## Why Selenium?
Currys, Argos and John Lewis are all **JavaScript-rendered React/Next.js apps**.  
Simple `requests` only gets an empty HTML shell — no products.  
Selenium launches a **real Chrome browser**, executes all JavaScript, waits for  
products to load, then reads the fully-populated DOM.

## Notebook Structure
1. Install packages
2. Configuration
3. Selenium browser setup
4. Product model + helpers

6. Argos scraper  

8. Run all scrapers
9. Clean + save results
10. Data quality report

---
## Cell 1 — Install Packages

In [1]:

# Verify
import selenium
print(f'   Selenium version: {selenium.__version__}')

   Selenium version: 4.41.0


---
## Cell 2 — ⚙️ Configuration

In [2]:
CONFIG = {
    # Category to scrape
    'category':         'laptops',   # 'laptops' or 'tablets'

    # Pages per retailer (each page ~24-30 products)
    # Start with 2 for testing, increase to 5+ for full dataset
    'max_pages': 5,

    # Headless mode:
    # True  = browser runs invisibly in background (faster)
    # False = you can SEE the browser open and scrape (good for debugging)
    'headless':          True,

    # Wait time for pages to load (seconds)
    # Increase if you get empty results on slow connections
    'page_load_wait':    8,
    'scroll_pause':      2,     # pause between scroll steps
    'scroll_steps':      5,     # how many times to scroll down per page

    # Polite delay between page navigations (ethical compliance)
    'delay_between_pages': 4,

    # Which retailers to scrape
    'scrape_currys':     False,
    'scrape_argos':      True,
    'scrape_johnlewis':  False,

    'output_dir':        './scraped_data',
}

import os
os.makedirs(CONFIG['output_dir'], exist_ok=True)
print(f" Config ready")
print(f"   Category  : {CONFIG['category']}")
print(f"   Max pages : {CONFIG['max_pages']} per retailer")
print(f"   Headless  : {CONFIG['headless']}")


 Config ready
   Category  : laptops
   Max pages : 5 per retailer
   Headless  : True


---
## Cell 3 — Selenium Browser Setup

In [3]:
import time, random, re, json, logging
import pandas as pd
from datetime import datetime
from dataclasses import dataclass, field, asdict
from typing import List, Optional
from bs4 import BeautifulSoup
from urllib.parse import urljoin

from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.webdriver.common.action_chains import ActionChains
from selenium.common.exceptions import (
    TimeoutException, NoSuchElementException,
    StaleElementReferenceException, WebDriverException
)
from webdriver_manager.chrome import ChromeDriverManager



def create_driver(headless: bool = True) -> webdriver.Chrome:
    """
    Create a Chrome WebDriver with anti-detection settings.
    webdriver-manager downloads the correct ChromeDriver automatically.
    """
    options = Options()

    if headless:
        options.add_argument('--headless=new')   

    # ── Anti-detection arguments ─────────────────────────────
    options.add_argument('--no-sandbox')
    options.add_argument('--disable-dev-shm-usage')
    options.add_argument('--disable-blink-features=AutomationControlled')
    options.add_argument('--disable-infobars')
    options.add_argument('--disable-extensions')
    options.add_argument('--disable-gpu')
    options.add_argument('--window-size=1920,1080')
    options.add_argument('--start-maximized')
    options.add_argument('--lang=en-GB')
    options.add_argument('--accept-lang=en-GB,en;q=0.9')

    # Realistic user agent
    options.add_argument(
        '--user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) '
        'AppleWebKit/537.36 (KHTML, like Gecko) '
        'Chrome/122.0.0.0 Safari/537.36'
    )

    # Remove automation flags
    options.add_experimental_option('excludeSwitches', ['enable-automation'])
    options.add_experimental_option('useAutomationExtension', False)

    # Preferences — accept cookies, UK locale
    prefs = {
        'intl.accept_languages':          'en-GB,en',
        'profile.default_content_setting_values.notifications': 2,
    }
    options.add_experimental_option('prefs', prefs)

    # Auto-download correct ChromeDriver
    service = Service(ChromeDriverManager().install())
    driver  = webdriver.Chrome(service=service, options=options)

    # Patch navigator.webdriver to hide Selenium
    driver.execute_script(
        "Object.defineProperty(navigator, 'webdriver', {get: () => undefined})"
    )
    driver.execute_cdp_cmd(
        'Network.setUserAgentOverride',
        {"userAgent": 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/122.0.0.0 Safari/537.36'}
    )

    
    return driver


# Quick test
print('Testing Chrome driver...')
try:
    test_driver = create_driver(headless=True)
    test_driver.get('https://www.google.com')
    print(f'Chrome works! Page title: {test_driver.title}')
    test_driver.quit()
except Exception as e:
    print(f'Chrome driver error: {e}')
    print('   Make sure Google Chrome is installed on your machine')
    print('   Download from: https://www.google.com/chrome/')

Testing Chrome driver...
Chrome works! Page title: Google


---
## Cell 4 — Product Model & Base Selenium Scraper

In [5]:
@dataclass
class Product:
    retailer:      str   = ''
    name:          str   = ''
    brand:         str   = ''
    model:         str   = ''
    price:         float = 0.0
    price_str:     str   = ''
    category:      str   = ''
    url:           str   = ''
    processor:     str   = ''
    ram:           str   = ''
    storage:       str   = ''
    screen_size:   str   = ''
    os:            str   = ''
    colour:        str   = ''
    rating:        str   = ''
    review_count:  str   = ''
    image_url:     str   = ''
    scraped_at:    str   = field(default_factory=lambda: datetime.now().isoformat())


class BaseSeleniumScraper:
    """Shared Selenium utilities for all retailer scrapers."""

    def __init__(self, retailer_name: str, base_url: str):
        self.retailer_name = retailer_name
        self.base_url      = base_url
        self.driver        = None
        self.products: List[Product] = []

    # ── Driver lifecycle ──────────────────────────────────────
    def start(self):
        self.driver = create_driver(headless=CONFIG['headless'])
       

    def stop(self):
        if self.driver:
            self.driver.quit()
            self.driver = None
           

    # ── Navigation ────────────────────────────────────────────
    def _go(self, url: str, wait_for_css: str = None, timeout: int = None):
        """
        Navigate to URL and wait for page to load.
        Optionally wait for a specific CSS selector to appear.
        """
        t = timeout or CONFIG['page_load_wait']
        try:
            self.driver.get(url)
            time.sleep(2)  # Initial load

            if wait_for_css:
                WebDriverWait(self.driver, t).until(
                    EC.presence_of_element_located((By.CSS_SELECTOR, wait_for_css))
                )
            else:
                # Wait for body to have content
                WebDriverWait(self.driver, t).until(
                    lambda d: len(d.find_elements(By.TAG_NAME, 'body')) > 0
                )

            print(f'[{self.retailer_name}] Loaded: {url[:70]}')
            return True

        except TimeoutException:
            print(f'[{self.retailer_name}] Timeout loading {url[:60]}')
            return False
        except WebDriverException as e:
            print(f'[{self.retailer_name}] WebDriver error: {e}')
            return False

    # ── Scrolling (triggers lazy-load) ────────────────────────
    def _scroll_page(self):
        """
        Scroll down the page in steps to trigger lazy-loading of products.
        Most retail sites only load product images/data when scrolled into view.
        """
        try:
            total_height = self.driver.execute_script('return document.body.scrollHeight')
            step         = total_height // CONFIG['scroll_steps']

            for i in range(CONFIG['scroll_steps']):
                scroll_to = step * (i + 1)
                self.driver.execute_script(f'window.scrollTo(0, {scroll_to});')
                time.sleep(CONFIG['scroll_pause'])

            # Scroll back to top
            self.driver.execute_script('window.scrollTo(0, 0);')
            time.sleep(1)
        except Exception as e:
            print(f'[{self.retailer_name}] Scroll error: {e}')

    # ── Cookie banner dismissal ───────────────────────────────
    def _dismiss_cookies(self):
        """
        Try to click cookie accept button.
        Sites show cookie banners that block content if not dismissed.
        """
        cookie_selectors = [
            # Generic
            'button[id*="accept"]',
            'button[class*="accept"]',
            'button[data-testid*="accept"]',
            '#onetrust-accept-btn-handler',
            '.cookie-accept',
            '[aria-label*="Accept"]',
            '[aria-label*="accept"]',
            # Currys specific
            'button[data-cc-action="accept"]',
            '#consent-accept',
            # John Lewis specific
            'button[data-test="allow-all-cookies"]',
            '#cookieBannerAccept',
            # Argos specific
            'button[id="consent_prompt_submit"]',
            '.consent-buttons button:first-child',
        ]
        for sel in cookie_selectors:
            try:
                btn = WebDriverWait(self.driver, 3).until(
                    EC.element_to_be_clickable((By.CSS_SELECTOR, sel))
                )
                btn.click()
                print(f'[{self.retailer_name}] Cookie banner dismissed ({sel})')
                time.sleep(1)
                return True
            except (TimeoutException, NoSuchElementException):
                continue
            except Exception:
                continue
        return False

    # ── Get rendered HTML ─────────────────────────────────────
    def _get_soup(self) -> BeautifulSoup:
        """Get BeautifulSoup from current page's rendered HTML."""
        return BeautifulSoup(self.driver.page_source, 'lxml')

    # ── Save debug HTML ───────────────────────────────────────
    def _save_debug(self, filename: str):
        path = f"{CONFIG['output_dir']}/{filename}_debug.html"
        with open(path, 'w', encoding='utf-8') as f:
            f.write(self.driver.page_source[:80000])
        print(f'Debug HTML saved → {path}')

    # ── Spec extractors ───────────────────────────────────────
    @staticmethod
    def _parse_price(s) -> float:
        try:
            return round(float(re.sub(r'[^\d.]', '', str(s).replace(',', '.'))), 2)
        except:
            return 0.0

    @staticmethod
    def _extract_brand(name: str) -> str:
        for b in ['Apple','Samsung','Dell','HP','Lenovo','ASUS','Acer',
                  'Microsoft','MSI','Razer','Google','Huawei','Honor','Sony','LG']:
            if b.lower() in name.lower():
                return b
        return name.split()[0].title() if name else ''

    @staticmethod
    def _extract_ram(t: str) -> str:
        m = re.search(r'(\d+)\s*GB\s*RAM', t, re.I) or re.search(r'(\d+)\s*GB', t, re.I)
        return f'{m.group(1)}GB' if m else ''

    @staticmethod
    def _extract_storage(t: str) -> str:
        m = re.search(r'(\d+)\s*(TB|GB)\s*(SSD|HDD|eMMC|NVMe)?', t, re.I)
        return f'{m.group(1)}{m.group(2).upper()}' if m else ''

    @staticmethod
    def _extract_screen(t: str) -> str:
        m = (re.search(r'(\d+\.?\d*)\s*(?:inch|in\.)', t, re.I) or
             re.search(r'(\d+\.\d+)"', t))
        return f'{m.group(1)}"' if m else ''

    @staticmethod
    def _extract_processor(t: str) -> str:
        for kw in ['Intel Core Ultra','Apple M4','Apple M3','Apple M2','Apple M1',
                   'AMD Ryzen','Intel Core i','Intel Celeron','Snapdragon']:
            if kw.lower() in t.lower():
                m = re.search(rf'{re.escape(kw)}[\w\s®™-]{{2,20}}', t, re.I)
                return m.group(0).strip() if m else kw
        return ''

    def _enrich(self, p: Product, text: str) -> Product:
        if not p.ram:         p.ram         = self._extract_ram(text)
        if not p.storage:     p.storage     = self._extract_storage(text)
        if not p.screen_size: p.screen_size = self._extract_screen(text)
        if not p.processor:   p.processor   = self._extract_processor(text)
        if not p.brand:       p.brand       = self._extract_brand(p.name)
        return p

    # ── Polite delay ──────────────────────────────────────────
    def _delay(self):
        time.sleep(random.uniform(
            CONFIG['delay_between_pages'],
            CONFIG['delay_between_pages'] + 2
        ))


print(' Base Selenium scraper ready')

 Base Selenium scraper ready


---
## Cell 6 — Argos Selenium Scraper

In [6]:
class ArgosSeleniumScraper(BaseSeleniumScraper):

    BASE_URL = 'https://www.argos.co.uk'
    SEARCH_URL = 'https://www.argos.co.uk/search/{term}/opt/page:{page}/'

    PRODUCT_SELECTORS = [
        'div[data-test="component-product-card"]',
        'article[class*="Product"]',
        'div[class*="product-card"]',
        'li[data-test*="product"]',
        '[data-component="ProductCard"]',
    ]

    def __init__(self):
        super().__init__('Argos', self.BASE_URL)

    def scrape(self, category='laptops', max_pages=2) -> List[Product]:
        print(f'\n{"="*55}\n Argos Selenium — {category}\n{"="*55}')

        self.start()
        try:
            for page in range(1, max_pages + 1):
                url = self.SEARCH_URL.format(term=category, page=page)
                print(f'[Argos] Page {page}/{max_pages} → {url}')

                ok = self._go(url)
                if not ok:
                    continue

                if page == 1:
                    self._dismiss_cookies()
                    time.sleep(2)

                # Wait for product grid
                self._wait_for_products()
                self._scroll_page()

                page_products = self._parse_page(category)
                self.products.extend(page_products)
                print(f'[Argos] Page {page}: {len(page_products)} | Total: {len(self.products)}')

                if not page_products:
                    self._save_debug(f'argos_page{page}')
                    if page > 1:
                        break

                self._delay()

        finally:
            self.stop()

        print(f'[Argos]  Done — {len(self.products)} products')
        return self.products

    def _wait_for_products(self):
        for sel in self.PRODUCT_SELECTORS:
            try:
                WebDriverWait(self.driver, CONFIG['page_load_wait']).until(
                    EC.presence_of_element_located((By.CSS_SELECTOR, sel))
                )
                return
            except TimeoutException:
                continue
        time.sleep(3)

    def _parse_page(self, category: str) -> List[Product]:
        soup     = self._get_soup()
        products = []

        items = []
        for sel in self.PRODUCT_SELECTORS:
            items = soup.select(sel)
            if items:
                print(f'[Argos] {len(items)} cards via "{sel}"')
                break

        if not items:
            return []

        for item in items:
            try:
                p = Product(retailer='Argos', category=category)

                # Name
                for sel in ['[data-test="product-title"]','h2','h3','[class*="Title"]','[class*="title"]']:
                    el = item.select_one(sel)
                    if el and el.get_text(strip=True):
                        p.name = el.get_text(strip=True)
                        break

                # Price
                for sel in ['[data-test="product-price"]','[class*="price"]','[class*="Price"]','strong']:
                    el = item.select_one(sel)
                    if el:
                        txt = el.get_text(strip=True)
                        if '£' in txt or re.search(r'\d{2,}', txt):
                            p.price_str = txt
                            p.price     = self._parse_price(txt)
                            break

                # URL
                link = item.select_one('a[href]')
                if link:
                    p.url = urljoin(self.BASE_URL, link['href'])

                # Rating
                rating_el = item.select_one('[class*="rating"], [aria-label*="star"]')
                if rating_el:
                    p.rating = rating_el.get('aria-label','') or rating_el.get_text(strip=True)

                # Image
                img = item.select_one('img')
                if img:
                    p.image_url = img.get('src','') or img.get('data-src','')

                p = self._enrich(p, item.get_text(' ', strip=True))

                if p.name:
                    products.append(p)
            except Exception as e:
                print(f'[Argos] Item error: {e}')

        return products


print(' ArgosSeleniumScraper ready')

 ArgosSeleniumScraper ready


---
## Cell 8 —  Run All Scrapers
>  Chrome windows will open (or run silently if headless=True).  
> Each retailer takes 2–5 minutes depending on page count.

In [7]:
all_products = []
category     = CONFIG['category']
max_pages    = CONFIG['max_pages']

print(f' Starting Selenium scraping run')
print(f'   Category  : {category}')
print(f'   Max pages : {max_pages} per retailer')
print(f'   Headless  : {CONFIG["headless"]}')
print(f'   Started   : {datetime.now().strftime("%H:%M:%S")}')
print()

# ── Argos ────────────────────────────────────────────────────
if CONFIG['scrape_argos']:
    try:
        scraper = ArgosSeleniumScraper()
        prods   = scraper.scrape(category=category, max_pages=max_pages)
        all_products.extend(prods)
        print(f'Argos      : {len(prods)} products ')
    except Exception as e:
        print(f'Argos      : FAILED — {e}')



print(f'\n Total raw products: {len(all_products)}')
print(f'   Finished  : {datetime.now().strftime("%H:%M:%S")}')

 Starting Selenium scraping run
   Category  : laptops
   Max pages : 5 per retailer
   Headless  : True
   Started   : 13:45:57


 Argos Selenium — laptops
[Argos] Page 1/5 → https://www.argos.co.uk/search/laptops/opt/page:1/
[Argos] Loaded: https://www.argos.co.uk/search/laptops/opt/page:1/
[Argos] Cookie banner dismissed (button[id*="accept"])
[Argos] 64 cards via "div[data-test="component-product-card"]"
[Argos] Page 1: 64 | Total: 64
[Argos] Page 2/5 → https://www.argos.co.uk/search/laptops/opt/page:2/
[Argos] Loaded: https://www.argos.co.uk/search/laptops/opt/page:2/
[Argos] 64 cards via "div[data-test="component-product-card"]"
[Argos] Page 2: 64 | Total: 128
[Argos] Page 3/5 → https://www.argos.co.uk/search/laptops/opt/page:3/
[Argos] Loaded: https://www.argos.co.uk/search/laptops/opt/page:3/
[Argos] 64 cards via "div[data-test="component-product-card"]"
[Argos] Page 3: 64 | Total: 192
[Argos] Page 4/5 → https://www.argos.co.uk/search/laptops/opt/page:4/
[Argos] Loaded: https:/

---
## Cell 9 — Clean & Save

In [9]:
if not all_products:
    print('   No products scraped.')
    print('   Set headless=False in CONFIG and re-run to see browser behaviour.')
    print('   Check *_debug.html files in ./scraped_data/')
else:
    # Build DataFrame
    df = pd.DataFrame([asdict(p) for p in all_products])
    df = df[df['name'].str.strip().ne('')]

    # Clean types
    df['price'] = pd.to_numeric(df['price'], errors='coerce').fillna(0)
    for col in ['name','brand','processor','ram','storage','screen_size']:
        df[col] = df[col].fillna('').astype(str).str.strip().replace('nan','')

    df['price_valid'] = df['price'].between(50, 10000)
    df['product_id']  = df.apply(
        lambda r: f"{r['retailer'][:3].upper()}_{abs(hash(r['name']+r['retailer']))%100000:05d}",
        axis=1
    )

    # Save
    ts        = datetime.now().strftime('%Y%m%d_%H%M%S')
    csv_path  = f"./products_{category}_{ts}.csv"
    json_path = f"./products_{category}_{ts}.json"

    df.to_csv(csv_path,  index=False, encoding='utf-8-sig')
    df.to_json(json_path, orient='records', indent=2, force_ascii=False)

    print(f' CSV  saved: {csv_path}')
    print(f'JSON saved: {json_path}')
    print(f'\n Results by retailer:')
    print(df.groupby('retailer')[['name','price_valid']].agg({'name':'count','price_valid':'sum'}).rename(columns={'name':'total','price_valid':'with_price'}).to_string())

    print(f'\n Sample products:')
    for retailer in df['retailer'].unique():
        print(f'\n   {retailer}')
        for _, r in df[df['retailer']==retailer].head(3).iterrows():
            print(f'    • {r["name"][:65]}')
            print(f'      £{r["price"]} | {r["ram"]} | {r["storage"]} | {r["processor"][:35]}')

 CSV  saved: ./products_laptops_20260404_134903.csv
JSON saved: ./products_laptops_20260404_134903.json

 Results by retailer:
          total  with_price
retailer                   
Argos       320         320

 Sample products:

   Argos
    • HP 14-em0018na 14 Inch AMD Athlon 4GB 128GB Laptop - Black
      £239.0 | 4GB | 4GB | 
    • HP 15-fc0078na 15.6 Inch AMD Athlon 8GB 128GB Laptop - Black
      £259.0 | 8GB | 8GB | 
    • Lenovo IdeaPad Slim 3 14in MediaTek 8GB 128GB Chromebook
      £279.0 | 8GB | 8GB | 
